# CardioGuard AI
**Deep Learning-Based Heart Disease Prediction and Risk Assessment System**

Educational academic prototype. Not a medical diagnostic tool. New, severe, ongoing, or concerning chest pain, shortness of breath, pain spreading to arm/shoulder/jaw, fainting or other serious symptoms need urgent medical assessment. India emergency number: **112**. Never delay care based on a low score.

## Abstract & Objectives
Train and compare three tabular neural networks on a real labeled dataset; audit data; analyze features; evaluate independent test results; save artifacts to Drive; create an interactive Streamlit dashboard and temporary HTTPS tunnel.

## System Architecture
Real dataset → audit and cleaning → stratified train/validation/test → train-only preprocessing → MLP / deep FFNN / residual MLP → validation threshold → test metrics → artifacts → dashboard.

In [ ]:
!pip -q install tensorflow streamlit plotly scikit-learn joblib cloudflared openpyxl

In [ ]:
import os,sys,json,time,random,subprocess,socket,re,zipfile,shutil
import numpy as np,pandas as pd,tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,average_precision_score,confusion_matrix,roc_curve,precision_recall_curve
import matplotlib.pyplot as plt,seaborn as sns
from google.colab import drive
from IPython.display import display,HTML

In [ ]:
SEED=42
os.environ['PYTHONHASHSEED']=str(SEED
)
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

In [ ]:
print('TensorFlow:',tf.__version__,'GPU:',tf.config.list_physical_devices('GPU'))

In [ ]:
drive.mount('/content/drive')

In [ ]:
ROOT='/content/drive/MyDrive/CardioGuard_AI'
for folder in ['models','artifacts','dataset','dashboard','reports']: os.makedirs(os.path.join(ROOT,folder),exist_ok=True)
print(ROOT)

## Dataset acquisition
The classic Cleveland dataset has 14 columns including target `num`; 0 is negative and positive values are collapsed to 1. Verify target encodings and definitions if using a different file. This dataset does **not** contain all 17 dashboard measurements.

In [ ]:
DATA_URL='https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data'
DATA_PATH='/content/heart_data.csv'
try:
 import requests
 names=['age','sex','cp','trestbps','chol','fbs','restecg','thalach','exang','oldpeak','slope','ca','thal','num']
 raw=pd.read_csv(DATA_URL,header=None,names=names,na_values='?',timeout=30); raw.to_csv(DATA_PATH,index=False); print('UCI loaded',raw.shape)
except Exception as err:
 print('Download failed; upload labeled CSV or Excel:',err)
 from google.colab import files
 uploads=files.upload()
 if not uploads: raise RuntimeError('Dataset file required')
 DATA_PATH='/content/'+next(iter(uploads)); raw=pd.read_csv(DATA_PATH) if DATA_PATH.endswith('.csv') else pd.read_excel(DATA_PATH)
display(raw.head()); print(raw.shape); display(raw.dtypes); display(raw.isna().sum())

In [ ]:
# Identify target (edit mapping only if your dataset encoding differs)
target_candidates=['num','target','heart_disease','diagnosis','output','label']
TARGET=next((c for c in raw.columns if str(c).lower().strip() in target_candidates),None)
if TARGET is None: TARGET=input('Target column name: ').strip()
yraw=raw[TARGET]
if pd.api.types.numeric_dtype(yraw): y=(pd.to_numeric(yraw,errors='coerce')>0).astype(int)
else: y=yraw.astype(str).str.lower().str.strip().map({'0':0,'no':0,'negative':0,'healthy':0,'1':1,'yes':1,'positive':1,'disease':1})
if y.isna().any(): raise ValueError('Unmapped target values; inspect and explicitly map them.')
X=raw.drop(columns=[TARGET]).copy()
for c in X.select_dtypes(include='object').columns:
 v=pd.to_numeric(X[c],errors='coerce')
 if v.notna().mean()>=.8: X[c]=v
print('Target distribution',y.value_counts().to_dict()); print('Features',list(X.columns))

In [ ]:
# Cleaning and compatibility audit
X=X.replace({'?':np.nan,'unknown':np.nan,'Unknown':np.nan})
clean=pd.concat([X,y.rename('__target__')],axis=1).drop_duplicates().dropna(subset=['__target__']); X=clean.drop(columns='__target__'); y=clean['__target__'].astype(int)
compat=pd.DataFrame({'feature':list(X.columns),'in_dataset':[True]*X.shape[1]}); display(compat); display(X.isna().sum()); display(X.describe(include='all').T)

In [ ]:
# EDA: target and numeric distributions
y.value_counts().plot(kind='bar',title='Label distribution'); plt.show()
for c in X.select_dtypes(include=np.number).columns[:10]: X[c].hist(bins=20); plt.title(c); plt.show()
num_cols=list(X.select_dtypes(include=np.number).columns); cat_cols=[c for c in X.columns if c not in num_cols]

In [ ]:
# Stratified 70/15/15 split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.15,stratify=y,random_state=SEED)
X_train,X_val,y_train,y_val=train_test_split(X_train,y_train,test_size=.1764706,stratify=y_train,random_state=SEED)
num_cols=list(X_train.select_dtypes(include=np.number).columns); cat_cols=[c for c in X_train.columns if c not in num_cols]
num_pipe=Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())])
cat_pipe=Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('encode',OneHotEncoder(handle_unknown='ignore',sparse_output=False))])
pre=ColumnTransformer([('num',num_pipe,num_cols),('cat',cat_pipe,cat_cols)])
Xtr=pre.fit_transform(X_train).astype('float32'); Xva=pre.transform(X_val).astype('float32'); Xte=pre.transform(X_test).astype('float32')
print(Xtr.shape,Xva.shape,Xte.shape)

In [ ]:
# Model 1: MLP
def build_mlp(n):
 i=tf.keras.Input(shape=(n,)); x=tf.keras.layers.Dense(64,activation='relu')(i); x=tf.keras.layers.BatchNormalization()(x); x=tf.keras.layers.Dropout(.3)(x); x=tf.keras.layers.Dense(32,activation='relu')(x); x=tf.keras.layers.Dropout(.2)(x); o=tf.keras.layers.Dense(1,activation='sigmoid')(x); m=tf.keras.Model(i,o); m.compile('adam','binary_crossentropy',metrics=['accuracy']); return m
mlp=build_mlp(Xtr.shape[1]); mlp.summary()

In [ ]:
# Train MLP
def callbacks(name): return [tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True),tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss',patience=4,factor=.5),tf.keras.callbacks.ModelCheckpoint(ROOT+'/models/'+name+'.keras',monitor='val_loss',save_best_only=True)]
h_mlp=mlp.fit(Xtr,y_train,validation_data=(Xva,y_val),epochs=80,batch_size=32,callbacks=callbacks('mlp_model'),verbose=1)

In [ ]:
# Model 2: Deep Feedforward Neural Network
def build_deep(n):
 i=tf.keras.Input(shape=(n,)); x=i
 for units,drop in [(128,.35),(96,.3),(64,.25),(32,.2)]:
  x=tf.keras.layers.Dense(units,kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x); x=tf.keras.layers.BatchNormalization()(x); x=tf.keras.layers.Activation('relu')(x); x=tf.keras.layers.Dropout(drop)(x)
 o=tf.keras.layers.Dense(1,activation='sigmoid')(x); m=tf.keras.Model(i,o); m.compile('adam','binary_crossentropy',metrics=['accuracy']); return m
deep_model=build_deep(Xtr.shape[1]); deep_model.summary()
h_deep=deep_model.fit(Xtr,y_train,validation_data=(Xva,y_val),epochs=80,batch_size=32,callbacks=callbacks('deep_model'),verbose=1)

In [ ]:
# Model 3: Residual MLP
def resblock(x,units=64):
 shortcut=x; z=tf.keras.layers.Dense(units,kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x); z=tf.keras.layers.BatchNormalization()(z); z=tf.keras.layers.Activation('relu')(z); z=tf.keras.layers.Dropout(.25)(z); z=tf.keras.layers.Dense(units,kernel_regularizer=tf.keras.regularizers.l2(1e-4))(z); z=tf.keras.layers.BatchNormalization()(z)
 if int(x.shape[-1])!=units: shortcut=tf.keras.layers.Dense(units)(shortcut)
 return tf.keras.layers.Activation('relu')(tf.keras.layers.Add()([z,shortcut]))
def build_res(n):
 i=tf.keras.Input(shape=(n,)); x=tf.keras.layers.Dense(64,activation='relu')(i); x=resblock(x); x=resblock(x); x=tf.keras.layers.Dropout(.2)(x); o=tf.keras.layers.Dense(1,activation='sigmoid')(x); m=tf.keras.Model(i,o); m.compile('adam','binary_crossentropy',metrics=['accuracy']); return m
res_model=build_res(Xtr.shape[1]); res_model.summary()
h_res=res_model.fit(Xtr,y_train,validation_data=(Xva,y_val),epochs=80,batch_size=32,callbacks=callbacks('residual_mlp_model'),verbose=1)

In [ ]:
# Validation threshold; test metrics are held out
models={'MLP':mlp,'Deep FFNN':deep_model,'Residual MLP':res_model}; histories={'MLP':h_mlp,'Deep FFNN':h_deep,'Residual MLP':h_res}
def threshold(yv,p):
 ts=np.linspace(.2,.8,61); return float(ts[np.argmax([f1_score(yv,p>=t,zero_division=0) for t in ts])])
thresholds={}; metrics={}
for name,m in models.items():
 pv=m.predict(Xva,verbose=0).ravel(); th=threshold(y_val,pv); thresholds[name]=th; p=m.predict(Xte,verbose=0).ravel(); pred=p>=th; cm=confusion_matrix(y_test,pred); tn,fp,fn,tp=cm.ravel()
 metrics[name]={'accuracy':float(accuracy_score(y_test,pred)),'precision':float(precision_score(y_test,pred,zero_division=0)),'recall':float(recall_score(y_test,pred,zero_division=0)),'f1':float(f1_score(y_test,pred,zero_division=0)),'roc_auc':float(roc_auc_score(y_test,p)),'pr_auc':float(average_precision_score(y_test,p)),'sensitivity':float(tp/(tp+fn) if tp+fn else 0),'specificity':float(tn/(tn+fp) if tn+fp else 0),'confusion_matrix':cm.tolist(),'threshold':th}
print(pd.DataFrame(metrics).T)

In [ ]:
# Confusion matrices and ROC/PR curves
for name,m in models.items():
 p=m.predict(Xte,verbose=0).ravel(); cm=confusion_matrix(y_test,p>=thresholds[name]); sns.heatmap(cm,annot=True,fmt='d'); plt.title(name+' confusion matrix'); plt.show()
 plt.figure();
 for nm,mm in models.items():
  pp=mm.predict(Xte,verbose=0).ravel(); fpr,tpr,_=roc_curve(y_test,pp); plt.plot(fpr,tpr,label=nm)
 plt.plot([0,1],[0,1],'--'); plt.legend(); plt.title('ROC'); plt.show()
 plt.figure();
 for nm,mm in models.items():
  pp=mm.predict(Xte,verbose=0).ravel(); pr,rc,_=precision_recall_curve(y_test,pp); plt.plot(rc,pr,label=nm)
 plt.legend(); plt.title('PR'); plt.xlabel('Recall'); plt.ylabel('Precision'); plt.show()


In [ ]:
# Comparison and selection based on validation F1 only
metric_df=pd.DataFrame(metrics).T; display(metric_df.drop(columns=['confusion_matrix','threshold']))
metric_df.drop(columns=['confusion_matrix','threshold']).plot(kind='bar',figsize=(12,5)); plt.show()
best_name=max(models,key=lambda n:f1_score(y_val,models[n].predict(Xva,verbose=0).ravel()>=thresholds[n],zero_division=0)); best_model=models[best_name]; print('Best validation F1 model:',best_name)

In [ ]:
# Save model/preprocessing artifacts and dataset
import joblib
for nm,m in zip(['mlp_model','deep_model','residual_mlp_model'],[mlp,deep_model,res_model]): m.save(ROOT+'/models/'+nm+'.keras')
best_model.save(ROOT+'/models/best_model.keras'); joblib.dump(pre,ROOT+'/artifacts/preprocessing_pipeline.pkl')
json.dump({'features':list(X.columns),'target':TARGET,'numeric_features':num_cols,'categorical_features':cat_cols},open(ROOT+'/artifacts/feature_config.json','w'),indent=2)
json.dump(thresholds,open(ROOT+'/artifacts/model_thresholds.json','w'),indent=2)
json.dump({k:{kk:vv for kk,vv in v.items() if kk!='confusion_matrix'} for k,v in metrics.items()},open(ROOT+'/artifacts/model_metrics.json','w'),indent=2)
X.assign(target=y).to_csv(ROOT+'/dataset/cleaned_dataset.csv',index=False)
json.dump({'shape':list(X.shape),'features':list(X.columns),'target':TARGET,'class_counts':y.value_counts().to_dict(),'source':DATA_URL},open(ROOT+'/dataset/dataset_metadata.json','w'),indent=2)
print('Artifacts saved')

## Dashboard generator cell
The generated dashboard has seven sections and displays dataset-supported model features only. Clinical inputs absent from the dataset are not substituted with fabricated values. For the classic UCI dataset, assessment is blocked if model inputs cannot be collected reliably from the form.

In [ ]:
# Generate Streamlit app.py
APP=os.path.join(ROOT,'dashboard','app.py')
APP_CODE="import os,json,joblib,datetime\nimport streamlit as st,pandas as pd,numpy as np,plotly.express as px\nfrom tensorflow.keras.models import load_model\nBASE=os.path.dirname(os.path.dirname(os.path.abspath(__file__))); ART=BASE+'/artifacts'; DATA=BASE+'/dataset'; MOD=BASE+'/models'\nst.set_page_config(page_title='CardioGuard AI',page_icon='🫀',layout='wide')\nst.markdown('<style>.stApp{background:#081729;color:#eaf2ff}.block-container{padding-top:1.2rem}.stMetric{background:#10263d;border-radius:12px;padding:12px}</style>',unsafe_allow_html=True)\nst.title('🫀 CARDIOGUARD AI'); st.caption('Heart Health Screening & Analytics — academic prototype')\nst.error('Not a diagnosis. Never use a low score to delay care. New, severe, ongoing, or concerning symptoms need immediate assessment. India emergency: 112.')\n@st.cache_resource\ndef load_all():\n pre=joblib.load(ART+'/preprocessing_pipeline.pkl'); cfg=json.load(open(ART+'/feature_config.json')); metrics=json.load(open(ART+'/model_metrics.json')); th=json.load(open(ART+'/model_thresholds.json')); models={k:load_model(MOD+'/'+f) for k,f in [('MLP','mlp_model.keras'),('Deep FFNN','deep_model.keras'),('Residual MLP','residual_mlp_model.keras')]}; return pre,cfg,metrics,th,models\ntry: pre,cfg,metrics,thresholds,models=load_all()\nexcept Exception as e: st.error('Artifacts unavailable. Train notebook models first: '+str(e)); st.stop()\ntry: data=pd.read_csv(DATA+'/cleaned_dataset.csv')\nexcept: data=pd.DataFrame()\nfeatures=cfg['features']; page=st.sidebar.radio('Navigation',['Overview','Patient Assessment','Medical Data Analytics','Model Performance','Feature Analysis','Assessment History','Dataset Management']); model_name=st.sidebar.selectbox('Model',list(models)); model=models[model_name]; th=thresholds[model_name]\nif page=='Overview':\n a,b,c,d=st.columns(4); a.metric('Dataset records',len(data)); b.metric('Features',len(features)); c.metric('Model',model_name); d.metric('Target positives',int(data.target.sum()) if 'target' in data else 'N/A'); st.dataframe(pd.DataFrame(metrics).T);\n if 'target' in data: st.plotly_chart(px.histogram(data,x='target',title='Class balance'),use_container_width=True)\n st.info('Only dataset features were used to train these models. This dataset does not necessarily include all requested symptom/body indicators.')\nelif page=='Patient Assessment':\n symptoms=['Chest pain/tightness','Shortness of breath','Pain left arm/shoulder/jaw','Irregular heartbeat','Dizziness/fainting','Fatigue/weakness','Feet/ankle swelling']; selected=[s for s in symptoms if st.checkbox(s)]\n if any(s in selected for s in symptoms[:3]+[symptoms[4]]): st.error('Urgent warning: symptoms may require urgent assessment. Do not wait for this AI prediction. If severe, new, ongoing, or concerning seek emergency care now. India emergency number 112.')\n st.subheader('17 requested inputs (for data collection; unsupported features are not model inputs)'); st.caption('Symptoms above plus body indicators below. These fields are not automatically mapped to features that were absent from training.')\n with st.form('patient_inputs'):\n  cols=st.columns(3); sysbp=cols[0].number_input('Systolic BP mmHg',0,350,120); diabp=cols[1].number_input('Diastolic BP mmHg',0,250,80); ldl=cols[2].number_input('LDL mg/dL',0,1000,100); hdl=cols[0].number_input('HDL mg/dL',0,500,50); chol=cols[1].number_input('Total cholesterol mg/dL',0,1500,180); trig=cols[2].number_input('Triglycerides mg/dL',0,3000,120); glucose=cols[0].number_input('Fasting glucose mg/dL',0,1000,95); a1c=cols[1].number_input('HbA1c %',0.0,30.0,5.5); waist=cols[2].number_input('Waist circumference cm',0,300,85); height=cols[0].number_input('Height cm',0.0,300.0,165.0); weight=cols[1].number_input('Weight kg',0.0,500.0,60.0); bmi=weight/(height/100)**2 if height else 0; cols[2].metric('Calculated BMI',f'{bmi:.1f}'); family=cols[0].selectbox('Early family history',['Unknown','Yes','No']); age=cols[1].number_input('Age',0,120,30); sex=cols[2].selectbox('Sex',['Not specified','Female','Male','Other']); submitted=st.form_submit_button('ANALYZE HEART HEALTH')\n if submitted:\n  st.warning('Dataset compatibility: assessment inputs are not necessarily supported by this trained model. Prediction will proceed only if every feature is directly mapped safely.')\n  alias={'age':age,'sex':1 if sex=='Male' else 0 if sex=='Female' else None,'trestbps':sysbp,'chol':chol,'fbs':int(glucose>=120)}; row={}\n  for f in features:\n   if f in alias and alias[f] is not None: row[f]=alias[f]\n   else: row[f]=None\n  if any(v is None for v in row.values()): st.error('Prediction blocked: this model has input features that cannot be entered or mapped reliably in this assessment form. Do not replace missing values with fabricated measurements.')\n  else:\n   score=float(model.predict(pre.transform(pd.DataFrame([row])).astype('float32'),verbose=0).ravel()[0]); category='Elevated model-estimated score' if score>=th else 'Lower model-estimated score'; st.metric('Screening category',category); st.metric('Raw uncalibrated model score',f'{score:.3f}'); st.caption('Not a medical probability, diagnosis, or reassurance. Consult a qualified clinician.')\n   rec={'time':datetime.datetime.now().isoformat(),'model':model_name,'score':score,'category':category}; hist=BASE+'/reports/assessment_history.csv'; pd.DataFrame([rec]).to_csv(hist,mode='a',header=not os.path.exists(hist),index=False); st.download_button('Download report CSV',pd.DataFrame([rec]).to_csv(index=False),'assessment.csv')\nelif page=='Medical Data Analytics':\n if data.empty: st.warning('Dataset not found')\n else:\n  st.dataframe(data.head(100)); numeric=data.select_dtypes(include=np.number).columns.tolist(); chosen=st.multiselect('Plot columns',numeric,default=numeric[:min(5,len(numeric))]);\n  for f in chosen: st.plotly_chart(px.histogram(data,x=f,title=f'{f} distribution'),use_container_width=True)\n  if len(numeric)>1: st.plotly_chart(px.imshow(data[numeric].corr(),title='Correlation heatmap'),use_container_width=True)\n  st.download_button('Download filtered data',data.to_csv(index=False),'dataset.csv')\nelif page=='Model Performance': st.dataframe(pd.DataFrame(metrics).T); st.caption('Metrics are held-out test metrics; model/threshold selection was done using validation data.')\nelif page=='Feature Analysis': st.write('Feature list used for training:'); st.write(features); st.info('Association and permutation sensitivity do not imply causation. Detailed processed-feature permutation analysis is in the notebook.')\nelif page=='Assessment History':\n f=BASE+'/reports/assessment_history.csv'; st.dataframe(pd.read_csv(f) if os.path.exists(f) else pd.DataFrame());\n if os.path.exists(f): st.download_button('Download history',pd.read_csv(f).to_csv(index=False),'history.csv')\nelif page=='Dataset Management':\n u=st.file_uploader('Upload CSV or Excel to inspect',type=['csv','xlsx']);\n if u:\n  df=pd.read_csv(u) if u.name.endswith('.csv') else pd.read_excel(u); st.dataframe(df.head()); st.write('Missing values:',df.isna().sum()); st.warning('Inspection only. Retraining must be performed explicitly in the notebook after compatibility checks.')\n st.write('Model feature list:',features)\n"
open(APP,'w',encoding='utf-8').write(APP_CODE)
open(ROOT+'/dashboard/requirements.txt','w').write('streamlit\ntensorflow\npandas\nnumpy\nscikit-learn\nplotly\njoblib\nopenpyxl\n')
print('App and requirements generated:',APP)

In [ ]:
# Artifact check
expected=['models/best_model.keras','models/mlp_model.keras','models/deep_model.keras','models/residual_mlp_model.keras','artifacts/preprocessing_pipeline.pkl','artifacts/feature_config.json','artifacts/model_metrics.json','artifacts/model_thresholds.json','dataset/cleaned_dataset.csv','dashboard/app.py']
for f in expected: print(('OK' if os.path.exists(ROOT+'/'+f) else 'NOT YET'),f)

In [ ]:
# Launch dashboard directly from Colab. Wait for dynamically generated tunnel URL.
PORT=8501
streamlit_log=open('/content/cardio_streamlit.log','w')
streamlit_proc=subprocess.Popen([sys.executable,'-m','streamlit','run',APP,'--server.port',str(PORT),'--server.address','0.0.0.0','--server.headless','true'],stdout=streamlit_log,stderr=subprocess.STDOUT)
ready=False
for _ in range(35):
 try: s=socket.create_connection(('127.0.0.1',PORT),timeout=1); s.close(); ready=True; break
 except: time.sleep(1)
if not ready: raise RuntimeError('Streamlit startup failed; check /content/cardio_streamlit.log')
cloudflared=shutil.which('cloudflared')
if not cloudflared:
 cloudflared='/usr/local/bin/cloudflared'; subprocess.run(['wget','-q','-O',cloudflared,'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'],check=True); os.chmod(cloudflared,0o755)
tunnel_proc=subprocess.Popen([cloudflared,'tunnel','--url',f'http://127.0.0.1:{PORT}','--no-autoupdate'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
public_url=None
for _ in range(100):
 line=tunnel_proc.stdout.readline()
 if line:
  print(line.strip()); match=re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com',line)
  if match: public_url=match.group(0); break
 time.sleep(.2)
if not public_url: raise RuntimeError('No tunnel URL detected; review tunnel output.')
print('Dashboard RUNNING. Local:',f'http://127.0.0.1:{PORT}','Public:',public_url)
display(HTML(f'<a href="{public_url}" target="_blank">Open CardioGuard AI dashboard</a>'))

In [ ]:
# Display URL
if globals().get('public_url'): display(HTML(f'<a href="{public_url}">{public_url}</a>'))

In [ ]:
# Shutdown tunnel and Streamlit
for p in ['tunnel_proc','streamlit_proc']:
 proc=globals().get(p)
 if proc is not None:
  try: proc.terminate(); proc.wait(timeout=5)
  except Exception:
   try: proc.kill()
   except: pass
print('CardioGuard processes stopped.')

In [ ]:
# Export project ZIP (run after training)
zip_path='/content/CardioGuard_AI_Complete_Project.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
 for base,dirs,files in os.walk(ROOT):
  for fn in files: z.write(os.path.join(base,fn),arcname=os.path.relpath(os.path.join(base,fn),ROOT))
from google.colab import files
files.download(zip_path)

## Final Summary
Run notebook cells in order. Dataset features and target must be reviewed for compatibility. Model scores are uncalibrated and are not clinical probabilities. The public tunnel is temporary and ends when Colab or the tunnel process stops. Avoid patient-identifying information.